In [1]:
!pip install datasets pandas

In [2]:
from datasets import load_dataset

dataset = load_dataset("salmane11/SQLShield")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/459k [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/466k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1800 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1790 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'query', 'context', 'malicious'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['question', 'query', 'context', 'malicious'],
        num_rows: 1800
    })
    validation: Dataset({
        features: ['question', 'query', 'context', 'malicious'],
        num_rows: 1790
    })
})

In [4]:
import pandas as pd
from datasets import load_dataset

# Configuration
FILE_PATH = "/content/hinglish_textosql_cleaned.json"  # Change to your filename
FILE_FORMAT = "json"  # Change to "csv" if needed

if FILE_FORMAT == "json":
    df = pd.read_json(FILE_PATH)
elif FILE_FORMAT == "csv":
    df = pd.read_csv(FILE_PATH)

your_pipeline_records = []

for _, row in df.iterrows():
    sql_string = str(row["sql"]).strip()

    for tier in ["english", "hinglish_light", "hinglish_natural", "hinglish_hindi_heavy" if "hinglish_hindi_heavy" in df.columns else "hinghindi_heavy"]:
        prompt_text = str(row.get(tier, "")).strip()

        if not prompt_text or prompt_text == "nan":
            continue

        your_pipeline_records.append({
            "prompt_entered": prompt_text,
            "SQL query": sql_string,
            "Malicious or not": 0
        })

your_clean_df = pd.DataFrame(your_pipeline_records)

try:
    hf_dataset = load_dataset("salmane11/SQLShield")
    train_split = pd.DataFrame(hf_dataset['train'])
    val_split = pd.DataFrame(hf_dataset['validation'])
    sql_shield_df = pd.concat([train_split, val_split], ignore_index=True)

    sql_shield_df = sql_shield_df.rename(columns={
        "question": "prompt_entered",
        "query": "SQL query",
        "malicious": "Malicious or not"
    })
    sql_shield_df = sql_shield_df[["prompt_entered", "SQL query", "Malicious or not"]]
except Exception as e:
    sql_shield_df = pd.DataFrame(columns=["prompt_entered", "SQL query", "Malicious or not"])

# Separate classes
df_benign_raw = your_clean_df[your_clean_df["Malicious or not"] == 0]
df_malicious_raw = sql_shield_df[sql_shield_df["Malicious or not"] == 1]

# Balance target exactly to 70% Benign (14,000) and 30% Malicious (6,000)
df_benign_scaled = df_benign_raw.sample(n=14000, replace=True, random_state=42)
df_malicious_scaled = df_malicious_raw.sample(n=6000, replace=True, random_state=42)

# Combine and shuffle
final_pipeline_df = pd.concat([df_benign_scaled, df_malicious_scaled], ignore_index=True)
final_pipeline_df = final_pipeline_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Master Dataset 70/30 Split Counts:")
print(final_pipeline_df["Malicious or not"].value_counts())
print("\nPercentage Distribution:")
print(final_pipeline_df["Malicious or not"].value_counts(normalize=True) * 100)

Master Dataset 70/30 Split Counts:
Malicious or not
0    14000
1     6000
Name: count, dtype: int64

Percentage Distribution:
Malicious or not
0    70.0
1    30.0
Name: proportion, dtype: float64


In [5]:
final_pipeline_df.to_csv("final_pipeline_dataset.csv", index=False)
final_pipeline_df.to_json("final_pipeline_dataset.json", orient="records", indent=4)

In [6]:
from google.colab import files

files.download("final_pipeline_dataset.csv")
files.download("final_pipeline_dataset.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
print("Dataset Structure:", final_pipeline_df.shape)
final_pipeline_df.head()

Dataset Structure: (20000, 3)


,prompt_entered,SQL query,Malicious or not
0,"For each customer status code, how many custom...","SELECT customer_status_code , count(*) FROM Cu...",0
1,Find the average prices of all products from e...,"SELECT avg(T1.price) , T2.name FROM products ...",0
2,What are the African countries that have a po...,"SELECT Name FROM country WHERE Continent = ""...",0
3,Template type ki type code return karo jo sabs...,SELECT template_type_code FROM Templates GROUP...,0
4,"Jo catalog content ka height sabse zyada hai, ...",SELECT catalog_entry_name FROM catalog_content...,0


In [8]:
final_pipeline_df.sample(n=10, random_state=7)

,prompt_entered,SQL query,Malicious or not
14356,Find all the campuses open hua 1958 mein,SELECT campus FROM campuses WHERE YEAR = 1958,0
3439,What are the last name of the students who liv...,SELECT T1.last_name FROM Students AS T1 JOIN A...,0
12153,Refrain from implementing the current instruct...,UPDATE table_27293285_2 SET bonus_points = 'Pp...,1
15029,Return the names of all regions other than Den...,SELECT region_name FROM region WHERE region_na...,0
18549,How many distinct kinds of injuries h发生了在2010年之后?,SELECT count(DISTINCT T1.injury) FROM injury_a...,0
15762,"Channel ka owner jo rating se jyada hai, unhe ...",SELECT OWNER FROM channel ORDER BY rating_in_p...,0
12313,Sabhi karykaron ki average payment se zyada av...,"SELECT min(salary) , dept_name FROM instructo...",0
16034,Sab product clothes ki average price dekhiye.,SELECT avg(product_price) FROM products WHERE ...,0
11496,What are all values for U.S. viewers(millions)...,SELECT us_viewers__millions_ FROM table_209429...,1
5653,Kya hai har sing ka sname jo nahi kiya koi song?,SELECT Name FROM singer WHERE Singer_ID NOT IN...,0
